# 06 - Final model sweep: best achievable Trip Validity classifier

A one-off, heavier search than the active-learning app's own routine
retrains (`ml/trip_validity_model/app/training.py::run_milestone`,
40 Optuna trials / 90s, one LightGBM config, one train/calibration/test
split) - now that the full 500-label budget is spent, this notebook
compares five model families with a much bigger search each, and
reports every diagnostic (ROC, precision-recall, confusion matrix,
calibration curve, feature importances) against a slice of data none of
the searching or calibrating ever touched.

**This notebook is read-only against Postgres and writes nothing back**
- no `INSERT`/`UPDATE`/`ALTER`, no touching
`ml/trip_validity_model/artifacts/` (the live app's own directory). The
one optional exception, clearly marked near the end, saves a model file
to a separate `06_artifacts/` folder for your own use - skip that cell
if you don't want even that.

## Why the data handling here isn't a plain train/calibration/test split

`ml.trip_validity_labels` isn't one clean random sample. Confirmed live
against the DB before writing this:

| label_set | selection_source | count |
|---|---|---|
| calibration | random | 125 |
| test | random | 125 |
| train | random | 126 |
| train | uncertain | 124 |

So there are **376 genuinely random-sourced rows** (not just the 250 in
calibration+test - `train` itself is half random draws, half
active-learning "most uncertain" picks) and **124 uncertain-sourced
rows**. The uncertain rows are deliberately boundary-enriched - exactly
what makes active learning effective for *training* a sharper decision
boundary - but they are not a representative sample of the real
population, so they must never be used to *measure* or *calibrate*
anything, only to help fit the classifier's own decision function.

The rule followed everywhere below: **train on everything, but only
ever validate, calibrate, or report a headline number using the
376-row clean (random-sourced) pool.**

Concretely, the clean pool gets split three ways (each split
stratified by label, fixed random seeds throughout for
reproducibility):

```text
clean pool (376, random-sourced)
├── search_pool (~300, 80%) -> Optuna hyperparameter/feature search,
│                               via its own internal 5-fold CV
└── final_holdout (~76, 20%) -> touched exactly once, at the very end,
                                 for the honest headline numbers + plots

search_pool further splits, only for the *final* refit of the winner
(not used during the search itself, which already cross-validates
over the whole search_pool):
├── final_train (~240, 80%) -> combined with the 124 uncertain rows to
│                               fit the winning model's decision function
└── final_calib (~60, 20%)  -> fits the probability calibrator, using
                                predictions from a model that never saw
                                these rows during training
```

This avoids calibration leakage: if the calibrator were fit on rows the
base model was trained on, its probability mapping would look better
than it really is.

In [ ]:
import os
import sys
import time
from collections.abc import Callable, Iterator
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import optuna
import pandas as pd
import psycopg
from psycopg import sql
from sklearn.calibration import CalibrationDisplay
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    PrecisionRecallDisplay,
    RocCurveDisplay,
)
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

optuna.logging.set_verbosity(optuna.logging.WARNING)  # one line per trial is too noisy

In [ ]:
_root = Path.cwd()
while not (_root / "pyproject.toml").exists():
    _root = _root.parent
os.chdir(_root)
os.environ.setdefault("RAW_DATA_ROOT", str(_root))
sys.path.insert(0, str(_root / "ml/trip_validity_model/app"))

## Connect

Reuses the app's own `features.py` (feature list + dtype casting),
`metrics.py` (AUC/Brier/log-loss/ECE, identical definitions to what the
app itself reports), and `calibration.py` (`PlattCalibrator`) directly -
they're plain, DB-independent utility modules. Everything that actually
*talks* to the database is reimplemented here as a fresh read-only
query rather than importing `db.py`, so this notebook has zero
dependency on the live app's own connection/session assumptions.

In [ ]:
import features
import metrics
from calibration import PlattCalibrator

from opa_database.config import settings

conn = psycopg.connect(settings.db_dsn)


def prepare_features(df: pd.DataFrame) -> pd.DataFrame:
    """Cast feature dtypes, then recode boolean categoricals to int categories.

    XGBoost's native categorical support (enable_categorical=True)
    rejects category dtypes whose *categories* are Python bool - only
    str/int categories are accepted - while LightGBM and CatBoost have
    no such restriction. Recoding False/True to 0/1 keeps every
    family's semantics identical (still a 2-level unordered
    categorical, NaN preserved) while satisfying XGBoost's constraint.
    """
    df = features.cast_feature_dtypes(df)
    for col in df.select_dtypes(include="category").columns:
        if pd.api.types.is_bool_dtype(df[col].cat.categories):
            df[col] = df[col].cat.rename_categories({False: 0, True: 1})
    return df


print("connected, read-only from here on")
print(
    f"{len(features.ALL_FEATURES)} candidate features "
    f"({len(features.CATEGORICAL_FEATURES)} categorical, "
    f"{len(features.NUMERIC_FEATURES)} numeric)"
)

## Load labeled data and the remaining unlabeled pool

Both queries select the exact same `features.ALL_FEATURES` list, built
via safe `sql.Identifier` composition (matching this project's own
convention) rather than string interpolation.

In [ ]:
feature_columns_sql = sql.SQL(", ").join(
    sql.SQL("d.{}").format(sql.Identifier(c)) for c in features.ALL_FEATURES
)

labeled_query = sql.SQL("""
    SELECT l.trip_id, l.label, l.label_set, l.selection_source, {feature_columns}
    FROM ml.trip_validity_labels l
    JOIN ml.trip_validity_dataset d ON d.trip_id = l.trip_id
""").format(feature_columns=feature_columns_sql)

with conn.cursor() as cur:
    cur.execute(labeled_query)
    cols = [c.name for c in cur.description]
    labeled = pd.DataFrame(cur.fetchall(), columns=cols)

labeled = prepare_features(labeled)
print(f"{len(labeled)} labeled rows")
print(labeled.groupby(["label_set", "selection_source"]).size())

In [ ]:
unlabeled_query = sql.SQL("""
    SELECT d.trip_id, {feature_columns}
    FROM ml.trip_validity_dataset d
    WHERE NOT EXISTS (
        SELECT 1 FROM ml.trip_validity_labels l WHERE l.trip_id = d.trip_id
    )
""").format(feature_columns=feature_columns_sql)

with conn.cursor() as cur:
    cur.execute(unlabeled_query)
    cols = [c.name for c in cur.description]
    unlabeled = pd.DataFrame(cur.fetchall(), columns=cols)

unlabeled = prepare_features(unlabeled)
print(f"{len(unlabeled)} unlabeled rows remaining")

EXPECTED_LABELED_COUNT = 500
if len(labeled) != EXPECTED_LABELED_COUNT:
    msg = f"expected {EXPECTED_LABELED_COUNT} labeled rows, got {len(labeled)}"
    raise AssertionError(msg)

## Build the pools

`label` comes back from Postgres as a Python `bool`; kept as-is (not
cast to 0/1) since every sklearn-compatible estimator used below
accepts boolean targets directly.

In [ ]:
SPLIT_SEED = 42

clean_pool = labeled[labeled["selection_source"] == "random"].reset_index(drop=True)
uncertain_pool = labeled[labeled["selection_source"] == "uncertain"].reset_index(
    drop=True
)

search_pool, final_holdout = train_test_split(
    clean_pool, test_size=0.2, stratify=clean_pool["label"], random_state=SPLIT_SEED
)
final_train, final_calib = train_test_split(
    search_pool, test_size=0.2, stratify=search_pool["label"], random_state=SPLIT_SEED
)

for name, pool in [
    ("clean_pool", clean_pool),
    ("uncertain_pool", uncertain_pool),
    ("search_pool", search_pool),
    ("final_holdout", final_holdout),
    ("final_train", final_train),
    ("final_calib", final_calib),
]:
    print(f"{name:15s} {len(pool):4d} rows, {pool['label'].mean():.1%} valid")

## Shared cross-validation harness

Every model family below calls this same `cv_score`: fit a fresh model
on each fold's training side, predict on the fold's validation side,
collect the 4 metrics `metrics.evaluate` already gives us. The
train/validation split itself comes from `cv_folds`, which is the one
place the "train on everything, validate on clean only" rule from the
intro actually gets enforced - every fold's training side is `(this
fold's slice of search_pool held out) -> everything else in
search_pool, plus every uncertain row`; every fold's validation side is
*only* a slice of `search_pool`. The uncertain rows never appear on the
validation side of any fold.

In [ ]:
N_CV_FOLDS = 5
CV_SEED = 42


def cv_folds(
    clean_df: pd.DataFrame, extra_train_df: pd.DataFrame, feature_cols: list[str]
) -> Iterator[tuple[pd.DataFrame, pd.Series, pd.DataFrame, pd.Series]]:
    """Yield (train_features, train_labels, valid_features, valid_labels) per fold.

    clean_df is split into N_CV_FOLDS folds; each fold's validation side
    is one fold's worth of clean_df, and its training side is the other
    folds of clean_df plus the *entire* extra_train_df (never held out).
    """
    skf = StratifiedKFold(n_splits=N_CV_FOLDS, shuffle=True, random_state=CV_SEED)
    clean_features = clean_df[feature_cols]
    clean_labels = clean_df["label"]
    extra_features = extra_train_df[feature_cols]
    extra_labels = extra_train_df["label"]
    for train_idx, valid_idx in skf.split(clean_features, clean_labels):
        train_features = pd.concat(
            [clean_features.iloc[train_idx], extra_features], ignore_index=True
        )
        train_labels = pd.concat(
            [clean_labels.iloc[train_idx], extra_labels], ignore_index=True
        )
        valid_features = clean_features.iloc[valid_idx]
        valid_labels = clean_labels.iloc[valid_idx]
        yield train_features, train_labels, valid_features, valid_labels


def cv_score(
    build_model: Callable[[], Any],
    feature_cols: list[str],
    clean_df: pd.DataFrame | None = None,
    extra_train_df: pd.DataFrame | None = None,
) -> pd.DataFrame:
    """Mean/std of each metric across cv_folds for one model configuration.

    build_model is a zero-argument callable returning a fresh, unfitted
    estimator (or sklearn Pipeline) each time it's called.
    """
    clean_df = search_pool if clean_df is None else clean_df
    extra_train_df = uncertain_pool if extra_train_df is None else extra_train_df
    fold_metrics = []
    for train_features, train_labels, valid_features, valid_labels in cv_folds(
        clean_df, extra_train_df, feature_cols
    ):
        model = build_model()
        model.fit(train_features, train_labels)
        proba = model.predict_proba(valid_features)[:, 1]
        fold_metrics.append(metrics.evaluate(valid_labels.to_numpy(), proba))
    return pd.DataFrame(fold_metrics)


def select_top_k_features(trial: optuna.Trial, ranking: list[str]) -> list[str]:
    """Let Optuna tune how many of the ranked features a trial gets to use."""
    top_k = trial.suggest_int("top_k", 10, len(ranking))
    return ranking[:top_k]

## Shared preprocessing for Random Forest and Logistic Regression

Unlike the three gradient-boosting families above, neither of these
handles `NaN` or pandas `category` columns natively - both need
imputation, categoricals need one-hot encoding, and Logistic Regression
additionally needs its numeric features scaled (it's sensitive to
feature magnitude in a way tree models never are). One shared builder,
parameterized by whether to scale, covers both - defined here, ahead of
feature ranking below, since Random Forest's and Logistic Regression's
own feature rankings need it too.

In [ ]:
def build_preprocessor(
    selected_features: list[str], *, scale_numeric: bool
) -> ColumnTransformer:
    """Build an imputer(+scaler)/one-hot ColumnTransformer for the given features."""
    cat_cols = [c for c in selected_features if c in features.CATEGORICAL_FEATURES]
    num_cols = [c for c in selected_features if c not in features.CATEGORICAL_FEATURES]

    numeric_steps = [("impute", SimpleImputer(strategy="median"))]
    if scale_numeric:
        numeric_steps.append(("scale", StandardScaler()))

    categorical_pipe = Pipeline(
        [
            ("impute", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore")),
        ]
    )

    return ColumnTransformer(
        [
            ("num", Pipeline(numeric_steps), num_cols),
            ("cat", categorical_pipe, cat_cols),
        ]
    )

### A shared feature-importance helper

Used twice: right below, to build Random Forest's and Logistic
Regression's own per-family feature ranking (rather than reusing
LightGBM's gain-based ranking, which reflects LightGBM's own splits,
not necessarily what a linear model or a bagged forest finds useful);
and again much further down, to plot the winning model's importances
against the holdout. Tree models expose `.feature_importances_`
directly; Logistic Regression is wrapped in a `Pipeline` with a
`ColumnTransformer` that one-hot-expands any categorical columns, so
its "importance" (absolute coefficient magnitude) comes back per
expanded column, not per original feature.

In [ ]:
def get_feature_importances(model: object, selected_features: list[str]) -> pd.Series:
    """Extract per-feature importance, handling tree models and LR Pipelines alike."""
    if hasattr(model, "feature_importances_"):
        return pd.Series(
            model.feature_importances_, index=selected_features
        ).sort_values(ascending=False)
    if isinstance(model, Pipeline):
        clf = model.named_steps["clf"]
        preprocessor = model.named_steps["preprocess"]
        feature_names = preprocessor.get_feature_names_out()
        if hasattr(clf, "coef_"):
            return pd.Series(np.abs(clf.coef_[0]), index=feature_names).sort_values(
                ascending=False
            )
    msg = f"don't know how to extract feature importances from {type(model)}"
    raise TypeError(msg)

## Rank features for the boosting families (LightGBM, XGBoost, CatBoost)

Same idea as the app's own `training.py::_rank_features_by_importance`
(mean CV gain importance from a quick LightGBM fit), reimplemented here
standalone rather than imported, so this notebook doesn't depend on a
private (underscore-prefixed) function from the app that could change
independently. Fit on `search_pool + uncertain_pool` (all data available
for training) - this is just a heuristic ordering to shrink each
model's `top_k` search space, not a performance measurement, so it
doesn't need to be clean-only.

Gain importance is a tree-splitting concept, so it's a reasonable proxy
across all three boosting families (they all pick thresholds to
maximize purity gain) - but not for Random Forest or Logistic
Regression, which don't rank features the same way a boosted tree
does. Those two get their own ranking next, built from their own kind
of model instead of borrowing this one.

In [ ]:
import lightgbm as lgb


def rank_features_by_importance(
    feature_matrix: pd.DataFrame, y: pd.Series, n_splits: int = 5, seed: int = 42
) -> list[str]:
    """Rank features by mean cross-validated LightGBM gain importance."""
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    importances = np.zeros(feature_matrix.shape[1])
    for train_idx, _ in skf.split(feature_matrix, y):
        ranker = lgb.LGBMClassifier(objective="binary", verbosity=-1, n_estimators=100)
        ranker.fit(feature_matrix.iloc[train_idx], y.iloc[train_idx])
        importances += ranker.feature_importances_
    order = np.argsort(importances)[::-1]
    return [feature_matrix.columns[i] for i in order]


_ranking_pool = pd.concat([search_pool, uncertain_pool], ignore_index=True)
FEATURE_RANKING = rank_features_by_importance(
    _ranking_pool[features.ALL_FEATURES], _ranking_pool["label"]
)
print("top 15 features by CV gain importance:")
for f in FEATURE_RANKING[:15]:
    print(" ", f)

### Rank features for Random Forest and Logistic Regression, using their own models

Borrowing LightGBM's gain ranking for these two would test them
somewhat handicapped: Random Forest's own splits and, especially,
Logistic Regression's linear coefficients can and do disagree with
LightGBM about which features matter most - a linear model cares about
clean separability, not split gain. Each gets one quick, untuned CV fit
of its own preprocess+model Pipeline (fixed placeholder
hyperparameters here; the real, tuned ones come from Optuna below),
using `get_feature_importances` and summing each categorical's
one-hot-expanded levels back onto that categorical's single ranking
entry.

In [ ]:
def rank_features_by_pipeline_importance(
    build_pipeline: Callable[[], Pipeline],
    feature_matrix: pd.DataFrame,
    y: pd.Series,
    feature_cols: list[str],
    n_splits: int = 5,
    seed: int = 42,
) -> list[str]:
    """Rank features via a fitted preprocess+model Pipeline's own importance.

    Complements rank_features_by_importance (LightGBM gain, used above
    for the 3 boosting families): Random Forest and Logistic Regression
    see one-hot-expanded columns through build_preprocessor, so their
    own importance signal lives there and gets summed back onto each
    original feature - a ranking shaped by how *this* family actually
    uses the data, not by a different family's splits.
    """
    cat_cols = set(features.CATEGORICAL_FEATURES)
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    totals = pd.Series(0.0, index=feature_cols)
    for train_idx, _ in skf.split(feature_matrix, y):
        pipeline = build_pipeline()
        pipeline.fit(feature_matrix.iloc[train_idx], y.iloc[train_idx])
        importances = get_feature_importances(pipeline, feature_cols)
        for expanded_name, value in importances.items():
            _, _, rest = expanded_name.partition("__")
            original_col = (
                rest
                if rest in totals.index
                else next(c for c in cat_cols if rest.startswith(f"{c}_"))
            )
            totals[original_col] += value
    return totals.sort_values(ascending=False).index.tolist()


FEATURE_RANKING_RF = rank_features_by_pipeline_importance(
    lambda: Pipeline(
        [
            (
                "preprocess",
                build_preprocessor(features.ALL_FEATURES, scale_numeric=False),
            ),
            ("clf", RandomForestClassifier(n_estimators=200, random_state=CV_SEED)),
        ]
    ),
    _ranking_pool[features.ALL_FEATURES],
    _ranking_pool["label"],
    features.ALL_FEATURES,
)
FEATURE_RANKING_LR = rank_features_by_pipeline_importance(
    lambda: Pipeline(
        [
            (
                "preprocess",
                build_preprocessor(features.ALL_FEATURES, scale_numeric=True),
            ),
            (
                "clf",
                LogisticRegression(
                    solver="liblinear", max_iter=2000, random_state=CV_SEED
                ),
            ),
        ]
    ),
    _ranking_pool[features.ALL_FEATURES],
    _ranking_pool["label"],
    features.ALL_FEATURES,
)

print("top 15 features for Random Forest:")
for f in FEATURE_RANKING_RF[:15]:
    print(" ", f)
print("top 15 features for Logistic Regression:")
for f in FEATURE_RANKING_LR[:15]:
    print(" ", f)

## Pull the existing production model's numbers, for reference

Not directly apples-to-apples - it was evaluated on its own 125-row
test set, a different specific slice than our `final_holdout` - but a
useful sanity-check baseline for "did this sweep actually do better
than what's already live."

In [ ]:
with conn.cursor() as cur:
    cur.execute("""
        SELECT run_id, created_at, run_type, n_train_labels,
               test_auc, test_brier, test_log_loss, test_ece
        FROM ml.trip_validity_model_runs
        ORDER BY created_at DESC
        LIMIT 1;
    """)
    cols = [c.name for c in cur.description]
    row = cur.fetchone()

production_baseline = dict(zip(cols, row, strict=True))
print("current production model (for reference only):")
for k, v in production_baseline.items():
    print(f"  {k}: {v}")

## Search intensity

Turn these up or down freely - this is the one knob controlling how
"absolute" the sweep actually is. Each of the 5 families below gets its
own `optuna.create_study`, stopping at whichever of `N_TRIALS`/
`TIMEOUT_SECONDS` comes first. On this dataset's size (a few hundred
rows per fold) even a few hundred trials per family should finish in
minutes, not hours - CatBoost is usually the slowest per-trial of the
five.

In [ ]:
N_TRIALS = 150
TIMEOUT_SECONDS = 300


def run_study(objective: Callable[[optuna.Trial], float], name: str) -> optuna.Study:
    """Run one Optuna TPE study and print a one-line timing/result summary."""
    study = optuna.create_study(
        direction="minimize", sampler=optuna.samplers.TPESampler(seed=CV_SEED)
    )
    start = time.monotonic()
    study.optimize(objective, n_trials=N_TRIALS, timeout=TIMEOUT_SECONDS)
    elapsed = time.monotonic() - start
    print(
        f"{name}: {len(study.trials)} trials in {elapsed:.0f}s, "
        f"best CV brier {study.best_value:.4f}"
    )
    return study

## LightGBM

The same model family the production app uses - included so the other
four families have something to actually beat, not just each other.

In [ ]:
def lightgbm_build_model(
    hyperparams: dict, selected_features: list[str]
) -> lgb.LGBMClassifier:
    """Build an unfit LightGBM classifier (uniform build_model(hp, feats) signature)."""
    del selected_features
    return lgb.LGBMClassifier(
        objective="binary", verbosity=-1, random_state=CV_SEED, **hyperparams
    )


def lightgbm_objective(trial: optuna.Trial) -> float:
    """Optuna objective: mean CV Brier score for one LightGBM configuration."""
    selected_features = select_top_k_features(trial, FEATURE_RANKING)
    hyperparams = {
        "num_leaves": trial.suggest_int("num_leaves", 7, 63),
        "min_child_samples": trial.suggest_int("min_child_samples", 5, 50),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "feature_fraction": trial.suggest_float("feature_fraction", 0.5, 1.0),
        "bagging_fraction": trial.suggest_float("bagging_fraction", 0.5, 1.0),
        "lambda_l1": trial.suggest_float("lambda_l1", 1e-8, 10.0, log=True),
        "lambda_l2": trial.suggest_float("lambda_l2", 1e-8, 10.0, log=True),
        "n_estimators": trial.suggest_int("n_estimators", 50, 400),
    }
    scores = cv_score(
        lambda: lightgbm_build_model(hyperparams, selected_features), selected_features
    )
    return float(scores["brier"].mean())


lightgbm_study = run_study(lightgbm_objective, "LightGBM")

## XGBoost

Same gradient-boosting family as LightGBM but a different growth
strategy and regularization defaults - `enable_categorical=True` +
`tree_method="hist"` let it use the pandas `category`-dtype columns
directly, same as LightGBM does automatically.

In [ ]:
import xgboost as xgb


def xgboost_build_model(
    hyperparams: dict, selected_features: list[str]
) -> xgb.XGBClassifier:
    """Build an unfit XGBoost classifier (uniform build_model(hp, feats) signature)."""
    del selected_features
    return xgb.XGBClassifier(
        objective="binary:logistic",
        eval_metric="logloss",
        enable_categorical=True,
        tree_method="hist",
        verbosity=0,
        random_state=CV_SEED,
        **hyperparams,
    )


def xgboost_objective(trial: optuna.Trial) -> float:
    """Optuna objective: mean CV Brier score for one XGBoost configuration."""
    selected_features = select_top_k_features(trial, FEATURE_RANKING)
    hyperparams = {
        "max_depth": trial.suggest_int("max_depth", 2, 8),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 20),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
        "n_estimators": trial.suggest_int("n_estimators", 50, 400),
    }
    scores = cv_score(
        lambda: xgboost_build_model(hyperparams, selected_features), selected_features
    )
    return float(scores["brier"].mean())


xgboost_study = run_study(xgboost_objective, "XGBoost")

## CatBoost

The one family here with real native categorical handling (ordered
target statistics, not just "split on category membership") - a direct
test of whether `company_id`/`weekday_number`/`is_weekend`/
`gtfs_route_has_both_directions` are actually useful once a model can
use them properly. The production LightGBM model dropped all four of
them from its top-70 feature selection.

In [ ]:
from catboost import CatBoostClassifier


def catboost_build_model(
    hyperparams: dict, selected_features: list[str]
) -> CatBoostClassifier:
    """Build an unfit CatBoost classifier, marking which features are categorical."""
    cat_features = [c for c in selected_features if c in features.CATEGORICAL_FEATURES]
    return CatBoostClassifier(
        cat_features=cat_features,
        verbose=False,
        allow_writing_files=False,  # no catboost_info/ dir left behind
        random_seed=CV_SEED,
        **hyperparams,
    )


def catboost_objective(trial: optuna.Trial) -> float:
    """Optuna objective: mean CV Brier score for one CatBoost configuration."""
    selected_features = select_top_k_features(trial, FEATURE_RANKING)
    hyperparams = {
        "depth": trial.suggest_int("depth", 3, 8),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1e-2, 10.0, log=True),
        "iterations": trial.suggest_int("iterations", 50, 400),
    }
    scores = cv_score(
        lambda: catboost_build_model(hyperparams, selected_features), selected_features
    )
    return float(scores["brier"].mean())


catboost_study = run_study(catboost_objective, "CatBoost")

## Random Forest

A bagging ensemble rather than boosting - a genuinely different
bias/variance tradeoff, and a useful check on whether the boosting
families are overfitting the small training pool.

In [ ]:
def random_forest_build_model(
    hyperparams: dict, selected_features: list[str]
) -> Pipeline:
    """Build an unfit preprocess+RandomForest Pipeline."""
    preprocessor = build_preprocessor(selected_features, scale_numeric=False)
    return Pipeline(
        [
            ("preprocess", preprocessor),
            ("clf", RandomForestClassifier(random_state=CV_SEED, **hyperparams)),
        ]
    )


def random_forest_objective(trial: optuna.Trial) -> float:
    """Optuna objective: mean CV Brier score for one Random Forest configuration."""
    selected_features = select_top_k_features(trial, FEATURE_RANKING_RF)
    hyperparams = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 600),
        "max_depth": trial.suggest_int("max_depth", 3, 20),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 20),
        "max_features": trial.suggest_float("max_features", 0.3, 1.0),
    }
    scores = cv_score(
        lambda: random_forest_build_model(hyperparams, selected_features),
        selected_features,
    )
    return float(scores["brier"].mean())


random_forest_study = run_study(random_forest_objective, "Random Forest")

## Regularized Logistic Regression

A linear baseline - `liblinear` supports both L1 and L2 penalty for
binary classification, and Optuna picks between them. Simple,
interpretable, and often surprisingly close to the tree-based models on
a dataset this size; also well-calibrated by construction, which is a
useful sanity check against the others.

In [ ]:
def logistic_regression_build_model(
    hyperparams: dict, selected_features: list[str]
) -> Pipeline:
    """Build an unfit preprocess+LogisticRegression Pipeline."""
    preprocessor = build_preprocessor(selected_features, scale_numeric=True)
    return Pipeline(
        [
            ("preprocess", preprocessor),
            (
                "clf",
                LogisticRegression(
                    solver="liblinear",
                    max_iter=2000,
                    random_state=CV_SEED,
                    **hyperparams,
                ),
            ),
        ]
    )


def logistic_regression_objective(trial: optuna.Trial) -> float:
    """Optuna objective: mean CV Brier score for one Logistic Regression config."""
    selected_features = select_top_k_features(trial, FEATURE_RANKING_LR)
    hyperparams = {
        "penalty": trial.suggest_categorical("penalty", ["l1", "l2"]),
        "C": trial.suggest_float("C", 1e-4, 10.0, log=True),
    }
    scores = cv_score(
        lambda: logistic_regression_build_model(hyperparams, selected_features),
        selected_features,
    )
    return float(scores["brier"].mean())


logistic_regression_study = run_study(
    logistic_regression_objective, "Logistic Regression"
)

## Compare all five

`split_best_params` separates Optuna's `top_k` from the model's own
tuned hyperparameters (both live in the same flat `study.best_params`
dict, since every `trial.suggest_*` call lands there together) - sliced
against whichever ranking that family's own search actually used
(`FEATURE_RANKING` for the 3 boosting families, `FEATURE_RANKING_RF`/
`FEATURE_RANKING_LR` for the other two), so each family's winning
`top_k` resolves against the same feature order it was tuned against.
Each family's winning config then gets one more `cv_score` pass to get
the full metric set (not just the Brier score Optuna optimized for)
with mean *and* std across folds - the std matters here as much as the
mean, given how few rows each fold has to work with, which is why
`selection_score` below blends the two rather than picking a winner on
mean alone.

In [ ]:
def split_best_params(
    study: optuna.Study, ranking: list[str]
) -> tuple[dict, list[str]]:
    """Split Optuna's flat best_params into (model hyperparams, selected features)."""
    params = dict(study.best_params)
    top_k = params.pop("top_k")
    return params, ranking[:top_k]


MODEL_FAMILIES = {
    "LightGBM": (lightgbm_build_model, lightgbm_study, FEATURE_RANKING),
    "XGBoost": (xgboost_build_model, xgboost_study, FEATURE_RANKING),
    "CatBoost": (catboost_build_model, catboost_study, FEATURE_RANKING),
    "Random Forest": (
        random_forest_build_model,
        random_forest_study,
        FEATURE_RANKING_RF,
    ),
    "Logistic Regression": (
        logistic_regression_build_model,
        logistic_regression_study,
        FEATURE_RANKING_LR,
    ),
}

results = {}
comparison_rows = []
for name, (build_fn, study, ranking) in MODEL_FAMILIES.items():
    hyperparams, selected_features = split_best_params(study, ranking)
    results[name] = {
        "build_fn": build_fn,
        "hyperparams": hyperparams,
        "selected_features": selected_features,
    }
    scores_df = cv_score(
        lambda hp=hyperparams, sf=selected_features, bf=build_fn: bf(hp, sf),
        selected_features,
    )
    row = {"model": name, "n_features": len(selected_features)}
    for metric_name in ["auc", "brier", "log_loss", "ece"]:
        row[f"{metric_name}_mean"] = scores_df[metric_name].mean()
        row[f"{metric_name}_std"] = scores_df[metric_name].std()
    comparison_rows.append(row)

comparison_table = (
    pd.DataFrame(comparison_rows).set_index("model").sort_values("brier_mean")
)

# Weight on brier_std when picking a winner below; 0 would mean-only.
# With ~5 folds over a few hundred rows each, a model whose mean looks
# best but whose std is wide got lucky on this particular fold split
# more than it demonstrated a real edge - this penalizes that.
SELECTION_STD_PENALTY = 1.0
comparison_table["selection_score"] = (
    comparison_table["brier_mean"]
    + SELECTION_STD_PENALTY * comparison_table["brier_std"]
)
comparison_table.round(4)

In [ ]:
sorted_by_brier = comparison_table["brier_mean"].sort_values()

fig, ax = plt.subplots(figsize=(8, 5))
sorted_by_brier.plot.barh(
    ax=ax, xerr=comparison_table.loc[sorted_by_brier.index, "brier_std"]
)
ax.axvline(
    production_baseline["test_brier"],
    color="black",
    linestyle="--",
    label=(
        f"current production model (test_brier={production_baseline['test_brier']:.4f})"
    ),
)
ax.set_xlabel("mean CV Brier score, +/- 1 std across folds (lower is better)")
ax.set_title("Model family comparison - stage-1 search CV, on the clean pool only")
ax.legend()
plt.tight_layout()
plt.show()

## The winner, evaluated once on data nothing above has ever seen

Picked by `selection_score` (mean + std penalty), not raw mean alone -
see the comparison cell above. The top 3 are printed below so you can
eyeball the actual mean/std trade-off before trusting the automatic
pick; with folds this small it's worth a second look, not just a
rubber stamp.

Refit on `final_train + uncertain_pool` (never `final_calib` or
`final_holdout`), calibrate on `final_calib` (a set the base model
never trained on), then predict on `final_holdout` exactly once. This
is the honest, no-leakage headline number - not the stage-1 CV mean
above, which (correctly) drove model *selection*, but was still an
average over data this exact winning config was repeatedly evaluated
against during the search.

In [ ]:
print(
    "top 3 by selection_score (brier_mean + penalty * brier_std) - "
    "eyeball before trusting the pick:"
)
print(
    comparison_table.sort_values("selection_score")[
        ["brier_mean", "brier_std", "selection_score"]
    ]
    .head(3)
    .round(4)
)

winner_name = comparison_table["selection_score"].idxmin()
winner = results[winner_name]
winner_build_fn = winner["build_fn"]
winner_hyperparams = winner["hyperparams"]
winner_features = winner["selected_features"]

print(f"\nwinner: {winner_name}")
print(f"  {len(winner_features)} features, hyperparameters: {winner_hyperparams}")

base_model = winner_build_fn(winner_hyperparams, winner_features)
base_model.fit(
    pd.concat(
        [final_train[winner_features], uncertain_pool[winner_features]],
        ignore_index=True,
    ),
    pd.concat([final_train["label"], uncertain_pool["label"]], ignore_index=True),
)

raw_calib_proba = base_model.predict_proba(final_calib[winner_features])[:, 1]
calibrator = PlattCalibrator().fit(raw_calib_proba, final_calib["label"].to_numpy())

y_true_holdout = final_holdout["label"].to_numpy()
raw_holdout_proba = base_model.predict_proba(final_holdout[winner_features])[:, 1]
calibrated_holdout_proba = calibrator.predict(raw_holdout_proba)

holdout_metrics = metrics.evaluate(y_true_holdout, calibrated_holdout_proba)
print("\nhonest holdout metrics (never touched by search, training, or calibration):")
for k, v in holdout_metrics.items():
    print(f"  {k}: {v:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
RocCurveDisplay.from_predictions(y_true_holdout, calibrated_holdout_proba, ax=axes[0])
axes[0].set_title(f"{winner_name} - ROC curve (holdout, n={len(final_holdout)})")
PrecisionRecallDisplay.from_predictions(
    y_true_holdout, calibrated_holdout_proba, ax=axes[1]
)
axes[1].set_title(f"{winner_name} - Precision-Recall curve (holdout)")
plt.tight_layout()
plt.show()

In [ ]:
DECISION_THRESHOLD = 0.5  # predicted valid iff calibrated P(valid) >= this

predicted_label_holdout = calibrated_holdout_proba >= DECISION_THRESHOLD
fig, ax = plt.subplots(figsize=(5, 5))
ConfusionMatrixDisplay.from_predictions(
    y_true_holdout,
    predicted_label_holdout,
    display_labels=["Invalid", "Valid"],
    ax=ax,
)
ax.set_title(
    f"{winner_name} - Confusion matrix (holdout, threshold={DECISION_THRESHOLD})"
)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
CalibrationDisplay.from_predictions(
    y_true_holdout, calibrated_holdout_proba, n_bins=10, ax=ax, name=winner_name
)
ax.set_title(f"{winner_name} - Calibration curve (holdout)")
plt.tight_layout()
plt.show()

### Feature importances for the winner

Plotted against the holdout-evaluated model above, reusing the same
`get_feature_importances` helper defined earlier (it was also used
there to build Random Forest's and Logistic Regression's own feature
rankings).

In [ ]:
importances = get_feature_importances(base_model, winner_features)
fig, ax = plt.subplots(figsize=(8, max(4, len(importances.head(20)) * 0.3)))
importances.head(20).sort_values().plot.barh(ax=ax)
ax.set_title(f"{winner_name} - top 20 feature importances")
plt.tight_layout()
plt.show()

## Bonus: the actual model to keep, and what it says about the rest of the fleet

Standard practice once you have an honest performance read: retrain one
last time on *all* 500 labeled rows (including `final_holdout` now)
to squeeze out the last bit of signal for the model you actually keep.
Its own calibration below is fit on `clean_pool`, which it also trained
on - a little optimistic in isolation - so trust the holdout numbers
above as the honest performance estimate, not anything computed
directly from this bonus model.

In [ ]:
final_deployable_model = winner_build_fn(winner_hyperparams, winner_features)
final_deployable_model.fit(labeled[winner_features], labeled["label"])

final_deployable_calibrator = PlattCalibrator().fit(
    final_deployable_model.predict_proba(clean_pool[winner_features])[:, 1],
    clean_pool["label"].to_numpy(),
)
print(f"{winner_name} refit on all {len(labeled)} labeled rows")

In [ ]:
unlabeled_raw_proba = final_deployable_model.predict_proba(unlabeled[winner_features])[
    :, 1
]
unlabeled_calibrated_proba = final_deployable_calibrator.predict(unlabeled_raw_proba)

print(f"predictions over the full remaining unlabeled pool ({len(unlabeled):,} trips):")
for threshold in [0.7, 0.8, 0.9, 0.95]:
    confident = (unlabeled_calibrated_proba >= threshold) | (
        unlabeled_calibrated_proba <= 1 - threshold
    )
    print(
        f"  confident at {threshold:.2f}: {confident.sum():,} / {len(unlabeled):,} "
        f"({confident.mean():.1%})"
    )

fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(unlabeled_calibrated_proba, bins=50)
ax.set_xlabel("calibrated P(valid)")
ax.set_ylabel("trip count")
ax.set_title("Predicted probability distribution across the full unlabeled pool")
plt.tight_layout()
plt.show()

### Optional: save this model locally

Off by default - flip `SAVE_ARTIFACT` to `True` and re-run this cell if
you want to keep it. Saved to `06_artifacts/`, a separate folder from
the live app's own `ml/trip_validity_model/artifacts/`, so this never
gets confused with (or accidentally loaded by) the running app.

In [ ]:
SAVE_ARTIFACT = False

if SAVE_ARTIFACT:
    import joblib

    artifact_dir = _root / "ml/trip_validity_model/notebooks/06_artifacts"
    artifact_dir.mkdir(parents=True, exist_ok=True)
    artifact_name = winner_name.lower().replace(" ", "_")
    artifact_path = artifact_dir / f"{artifact_name}_{int(time.time())}.joblib"
    joblib.dump(
        {
            "model": final_deployable_model,
            "calibrator_params": final_deployable_calibrator.to_params(),
            "selected_features": winner_features,
            "hyperparameters": winner_hyperparams,
            "holdout_metrics": holdout_metrics,
        },
        artifact_path,
    )
    print(f"saved to {artifact_path}")
else:
    print("SAVE_ARTIFACT is False - nothing written to disk.")